# Notebook 0 — Validation: Building the Perceptron from Scratch

> **Story context:** Before shipping anything to CompanyX, we need to be confident our tools work correctly. This notebook validates each perceptron variant on simple, well-understood problems where we know the expected answer. Think of it as unit-testing our neural network implementation.

We validate four models in order of increasing complexity:
1. **Step Perceptron** → AND gate (binary classification)
2. **Linear Perceptron** → fitting y = x (regression)
3. **Non-linear Perceptron** → fitting y = tanh(x) (non-linear regression)
4. **Multilayer Perceptron** → XOR gate (non-linearly separable classification)

Each model is implemented **from scratch using NumPy only**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size'] = 11

---
## Part 1 — Step Perceptron

### Theory
The step perceptron computes:
$$O = \text{sign}\left(\sum_{i} w_i x_i\right)$$

The **Rosenblatt learning rule** (1958) updates weights on misclassification:
$$\Delta w_i = \eta \cdot (\zeta - O) \cdot x_i$$

**Key constraint:** only solves **linearly separable** problems.

In [ ]:
class StepPerceptron:
    """Simple step perceptron. Bias treated as extra weight with input=1."""

    def __init__(self, n_features, learning_rate=0.1, max_epochs=1000):
        self.lr = learning_rate
        self.max_epochs = max_epochs
        self.w = np.random.uniform(-1, 1, size=n_features + 1)
        self.history = []

    def _add_bias(self, X):
        return np.hstack([np.ones((X.shape[0], 1)), X])

    def _activation(self, h):
        return np.where(h >= 0, 1, -1)

    def predict(self, X):
        return self._activation(self._add_bias(X) @ self.w)

    def fit(self, X, y):
        X_b = self._add_bias(X)
        p = X_b.shape[0]
        best_w, best_error = self.w.copy(), p

        for epoch in range(self.max_epochs):
            idx = np.random.randint(p)
            o = self._activation(X_b[idx] @ self.w)
            self.w += self.lr * (y[idx] - o) * X_b[idx]

            error = np.sum(self._activation(X_b @ self.w) != y)
            self.history.append(error)
            if error < best_error:
                best_error, best_w = error, self.w.copy()
            if error == 0:
                print(f"  Converged at epoch {epoch+1}")
                break

        self.w = best_w
        return self

### Experiment: AND gate
AND is linearly separable — the step perceptron must converge to 0 errors.

In [ ]:
X_and = np.array([[-1, 1], [1, -1], [-1, -1], [1, 1]], dtype=float)
y_and = np.array([-1, -1, -1, 1], dtype=float)

model_step = StepPerceptron(n_features=2, learning_rate=0.1, max_epochs=500)
model_step.fit(X_and, y_and)

preds = model_step.predict(X_and)
print(f"  Predictions : {preds}")
print(f"  Expected    : {y_and}")
print(f"  Accuracy    : {np.mean(preds == y_and)*100:.0f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 300), np.linspace(-1.5, 1.5, 300))
Z = model_step.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contourf(xx, yy, Z, alpha=0.2, cmap='RdYlGn')
ax.contour(xx, yy, Z, colors='k', linewidths=1)
colors = ['#E24B4A' if yi == -1 else '#1D9E75' for yi in y_and]
ax.scatter(X_and[:, 0], X_and[:, 1], c=colors, s=120, edgecolors='k', zorder=5)
for xi, yi_label in zip(X_and, y_and):
    ax.annotate(f'  y={int(yi_label)}', xi, fontsize=9)
ax.set_title('AND gate — decision boundary')
ax.set_xlabel('x1'); ax.set_ylabel('x2')

axes[1].plot(model_step.history, color='#534AB7', lw=1.5)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Misclassified samples')
axes[1].set_title('Learning curve — step perceptron')

plt.tight_layout()
plt.savefig('val_step_perceptron.png', bbox_inches='tight')
plt.show()
print("The step perceptron finds the separating hyperplane and converges to 0 errors.")

---
## Part 2 — Linear Perceptron (ADALINE)

### Theory
Replace the step function with the identity. We minimize SSE:
$$E(\mathbf{w}) = \frac{1}{2} \sum_{\mu} (\zeta^\mu - O^\mu)^2$$

Gradient descent gives the **delta rule** (Widrow-Hoff, 1960):
$$\Delta w_i = \eta (\zeta^\mu - O^\mu) x_i^\mu$$

Same update rule as the step perceptron — but derived from a mathematical cost function.

In [ ]:
class LinearPerceptron:
    """ADALINE: identity activation, SSE loss, delta rule update."""

    def __init__(self, n_features, learning_rate=0.01, max_epochs=2000):
        self.lr = learning_rate
        self.max_epochs = max_epochs
        self.w = np.random.uniform(-1, 1, size=n_features + 1)
        self.loss_history = []

    def _add_bias(self, X):
        return np.hstack([np.ones((X.shape[0], 1)), X])

    def predict(self, X):
        return self._add_bias(X) @ self.w

    def fit(self, X, y, verbose_every=500):
        X_b = self._add_bias(X)
        p = X_b.shape[0]
        for epoch in range(self.max_epochs):
            idx = np.random.randint(p)
            o = X_b[idx] @ self.w
            self.w += self.lr * (y[idx] - o) * X_b[idx]
            sse = 0.5 * np.sum((y - X_b @ self.w) ** 2)
            self.loss_history.append(sse)
            if verbose_every and (epoch + 1) % verbose_every == 0:
                print(f"  Epoch {epoch+1:4d} | SSE = {sse:.4f}")
        return self

In [ ]:
X_lin = np.random.uniform(-3, 3, size=(50, 1))
y_lin = X_lin.ravel() + np.random.normal(0, 0.3, size=50)

model_lin = LinearPerceptron(n_features=1, learning_rate=0.005, max_epochs=2000)
model_lin.fit(X_lin, y_lin, verbose_every=500)
print(f"\n  Learned: bias={model_lin.w[0]:.3f}, w1={model_lin.w[1]:.3f}  (ideal: ~0, ~1)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x_plot = np.linspace(-3.5, 3.5, 100).reshape(-1, 1)

axes[0].scatter(X_lin, y_lin, alpha=0.6, s=30, color='#378ADD', label='Data')
axes[0].plot(x_plot, model_lin.predict(x_plot), color='#E24B4A', lw=2, label='Learned')
axes[0].plot(x_plot, x_plot, color='gray', lw=1.5, ls='--', label='True y=x')
axes[0].set_title('Linear perceptron — fitting y = x')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y'); axes[0].legend()

axes[1].plot(model_lin.loss_history, color='#534AB7', lw=1)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('SSE')
axes[1].set_title('SSE loss over training')

plt.tight_layout()
plt.savefig('val_linear_perceptron.png', bbox_inches='tight')
plt.show()
print("The delta rule converges to the correct regression line — implementation verified.")

---
## Part 3 — Non-linear Perceptron

### Theory
We compose the excitation with a nonlinear activation $g(\cdot)$:
$$O^\mu = g(h^\mu) = g\left(\sum_i w_i x_i^\mu\right)$$

The chain rule gives the weight update:
$$\Delta w_i = \eta (\zeta^\mu - O^\mu) \cdot g'(h^\mu) \cdot x_i^\mu$$

For $\tanh$: $g'(h) = \beta(1 - g(h)^2)$ — expressed using the output directly, no extra computation.

> **This chain rule is the embryo of backpropagation** — we will generalize it to multiple layers in Part 4.

In [ ]:
class NonLinearPerceptron:
    """Non-linear simple perceptron. Supports tanh, sigmoid, relu activations."""

    ACTIVATIONS = {
        'tanh':    (lambda h, b: np.tanh(b * h),             lambda g, b: b * (1 - g**2)),
        'sigmoid': (lambda h, b: 1/(1+np.exp(-2*b*h)),       lambda g, b: 2*b*g*(1-g)),
        'relu':    (lambda h, b: np.maximum(0, h),            lambda g, b: (g > 0).astype(float)),
    }

    def __init__(self, n_features, learning_rate=0.01, max_epochs=3000,
                 activation='tanh', beta=1.0):
        self.lr = learning_rate
        self.max_epochs = max_epochs
        self.beta = beta
        self.activation_name = activation
        self.g, self.g_prime = self.ACTIVATIONS[activation]
        self.w = np.random.uniform(-1, 1, size=n_features + 1)
        self.loss_history = []

    def _add_bias(self, X):
        return np.hstack([np.ones((X.shape[0], 1)), X])

    def predict(self, X):
        h = self._add_bias(X) @ self.w
        return self.g(h, self.beta)

    def fit(self, X, y, verbose_every=1000):
        X_b = self._add_bias(X)
        p = X_b.shape[0]
        for epoch in range(self.max_epochs):
            idx = np.random.randint(p)
            h = X_b[idx] @ self.w
            o = self.g(h, self.beta)
            grad = (y[idx] - o) * self.g_prime(o, self.beta)
            self.w += self.lr * grad * X_b[idx]
            preds = self.predict(X)
            self.loss_history.append(0.5 * np.sum((y - preds) ** 2))
            if verbose_every and (epoch + 1) % verbose_every == 0:
                print(f"  Epoch {epoch+1:5d} | SSE = {self.loss_history[-1]:.4f}")
        return self

In [ ]:
X_nl = np.random.uniform(-3, 3, size=(50, 1))
y_nl = np.tanh(X_nl.ravel()) + np.random.normal(0, 0.05, size=50)

model_tanh = NonLinearPerceptron(1, learning_rate=0.01, max_epochs=3000, activation='tanh')
model_tanh.fit(X_nl, y_nl, verbose_every=1000)

model_relu = NonLinearPerceptron(1, learning_rate=0.005, max_epochs=3000, activation='relu')
model_relu.fit(X_nl, y_nl, verbose_every=1000)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x_plot = np.linspace(-3.5, 3.5, 300).reshape(-1, 1)

ax = axes[0]
ax.scatter(X_nl, y_nl, alpha=0.5, s=30, color='#378ADD', label='Data', zorder=3)
ax.plot(x_plot, np.tanh(x_plot), color='gray', lw=1.5, ls='--', label='True tanh(x)')
ax.plot(x_plot, model_tanh.predict(x_plot), color='#1D9E75', lw=2, label='tanh perceptron')
ax.plot(x_plot, model_relu.predict(x_plot), color='#D85A30', lw=2, label='ReLU perceptron')
ax.set_title('Non-linear perceptron — fitting y = tanh(x)')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.legend(fontsize=9); ax.set_ylim(-1.5, 1.5)

axes[1].plot(model_tanh.loss_history, color='#1D9E75', lw=1.5, label='tanh')
axes[1].plot(model_relu.loss_history, color='#D85A30', lw=1.5, label='ReLU')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('SSE')
axes[1].set_title('SSE loss — tanh vs ReLU'); axes[1].legend()

plt.tight_layout()
plt.savefig('val_nonlinear_perceptron.png', bbox_inches='tight')
plt.show()
print("tanh perceptron fits the shape well (it IS the target function).")
print("ReLU can only produce piecewise linear output — rougher approximation of tanh.")

---
## Part 4 — Multilayer Perceptron (MLP) & Backpropagation

### Theory

Minsky & Papert (1969): the step perceptron fails on XOR (not linearly separable).  
Solution: **stack layers** (hierarchical agglomerative structure).

**Feedforward** for a [2, 2, 1] network:
$$V_j^\mu = g\left(\sum_k w_{jk} x_k^\mu\right) \qquad O_i^\mu = g\left(\sum_j W_{ij} V_j^\mu\right)$$

**Backpropagation** (Rumelhart, Hinton & Williams, 1986):
- Output delta: $\delta_i = (\zeta_i - O_i) \cdot g'(h_i)$
- Hidden delta: $\delta_j = g'(h_j) \cdot \sum_i W_{ij} \delta_i$
- Update: $\Delta W_{ij} = \eta \delta_i V_j$, $\quad \Delta w_{jk} = \eta \delta_j x_k$

In [ ]:
class MLP:
    """
    Multilayer Perceptron with arbitrary architecture.
    Activation: tanh for all layers.
    Training: online backpropagation (one sample per step).
    Architecture example: [2, 4, 1] = 2 inputs, 4 hidden, 1 output.
    """

    def __init__(self, architecture, learning_rate=0.1, max_epochs=5000, beta=1.0):
        self.lr = learning_rate
        self.max_epochs = max_epochs
        self.beta = beta
        self.arch = architecture
        self.n_layers = len(architecture) - 1
        # W[l]: shape (arch[l+1], arch[l]+1) — +1 for bias
        self.W = [
            np.random.uniform(-1, 1, size=(architecture[l+1], architecture[l] + 1))
            for l in range(self.n_layers)
        ]
        self.loss_history = []

    def _g(self, h):         return np.tanh(self.beta * h)
    def _gp(self, g):        return self.beta * (1 - g**2)  # g' expressed via g
    def _bias(self, a):      return np.concatenate([[1.0], a])

    def _forward(self, x):
        """Forward pass. Returns list of (h, g) per layer."""
        layers = []  # (h, a) per layer
        a = x
        for l in range(self.n_layers):
            h = self.W[l] @ self._bias(a)
            a = self._g(h)
            layers.append((h, a))
        return layers

    def predict_one(self, x):
        return self._forward(x)[-1][1]

    def predict(self, X):
        return np.array([self.predict_one(x) for x in X])

    def _backward(self, x, zeta):
        layers = self._forward(x)
        zeta = np.atleast_1d(zeta)
        n = self.n_layers
        deltas = [None] * n

        # Output layer delta
        h_out, a_out = layers[-1]
        deltas[-1] = (zeta - a_out) * self._gp(a_out)

        # Hidden layers — backpropagate
        for l in range(n - 2, -1, -1):
            h_l, a_l = layers[l]
            # W[l+1] without bias column
            deltas[l] = self._gp(a_l) * (self.W[l+1][:, 1:].T @ deltas[l+1])

        # Compute gradients
        grads = []
        for l in range(n):
            a_in = x if l == 0 else layers[l-1][1]
            grads.append(np.outer(deltas[l], self._bias(a_in)))

        return grads

    def fit(self, X, y, verbose_every=2000):
        p = X.shape[0]
        y = y.ravel()
        for epoch in range(self.max_epochs):
            idx = np.random.randint(p)
            for l, grad in enumerate(self._backward(X[idx], y[idx])):
                self.W[l] += self.lr * grad
            preds = self.predict(X)
            sse = 0.5 * np.sum((y - preds.ravel())**2)
            self.loss_history.append(sse)
            if verbose_every and (epoch+1) % verbose_every == 0:
                print(f"  Epoch {epoch+1:5d} | SSE = {sse:.4f}")
        return self

In [ ]:
X_xor = np.array([[-1, 1], [1, -1], [-1, -1], [1, 1]], dtype=float)
y_xor = np.array([1., 1., -1., -1.])

# Step perceptron on XOR — should fail
model_step_xor = StepPerceptron(n_features=2, learning_rate=0.1, max_epochs=500)
model_step_xor.fit(X_xor, y_xor)
print(f"Step perceptron on XOR: {np.mean(model_step_xor.predict(X_xor)==y_xor)*100:.0f}% (expected <100%)\n")

# MLP [2,2,1] — try multiple seeds, keep best
best_221, best_loss = None, np.inf
for seed in range(30):
    np.random.seed(seed)
    m = MLP([2, 2, 1], learning_rate=0.5, max_epochs=10000)
    m.fit(X_xor, y_xor, verbose_every=0)
    if m.loss_history[-1] < best_loss:
        best_loss, best_221 = m.loss_history[-1], m

p221 = best_221.predict(X_xor).ravel()
print(f"MLP [2,2,1]   XOR accuracy: {np.mean(np.sign(p221)==y_xor)*100:.0f}% | SSE: {best_loss:.4f}")
print(f"  Outputs: {p221.round(3)}")

# MLP [2,3,2,1]
best_2321, best_loss2 = None, np.inf
for seed in range(30):
    np.random.seed(seed)
    m = MLP([2, 3, 2, 1], learning_rate=0.3, max_epochs=10000)
    m.fit(X_xor, y_xor, verbose_every=0)
    if m.loss_history[-1] < best_loss2:
        best_loss2, best_2321 = m.loss_history[-1], m

p2321 = best_2321.predict(X_xor).ravel()
print(f"\nMLP [2,3,2,1] XOR accuracy: {np.mean(np.sign(p2321)==y_xor)*100:.0f}% | SSE: {best_loss2:.4f}")
print(f"  Outputs: {p2321.round(3)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 300), np.linspace(-1.5, 1.5, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
colors_pts = ['#E24B4A' if yi == -1 else '#1D9E75' for yi in y_xor]

for ax, (model, title) in zip(axes, [
    (model_step_xor, 'Step Perceptron (fails on XOR)'),
    (best_221,       'MLP [2,2,1]'),
    (best_2321,      'MLP [2,3,2,1]'),
]):
    preds_grid = np.sign(model.predict(grid).ravel()).reshape(xx.shape)
    ax.contourf(xx, yy, preds_grid, alpha=0.2, cmap='RdYlGn')
    ax.contour(xx, yy, preds_grid, colors='k', linewidths=1)
    ax.scatter(X_xor[:,0], X_xor[:,1], c=colors_pts, s=120, edgecolors='k', zorder=5)
    for xi, yi_label in zip(X_xor, y_xor):
        ax.annotate(f'  y={int(yi_label)}', xi, fontsize=9)
    ax.set_title(title); ax.set_xlabel('x1'); ax.set_ylabel('x2')

plt.suptitle('XOR — decision boundaries', fontsize=13)
plt.tight_layout()
plt.savefig('val_mlp_xor.png', bbox_inches='tight')
plt.show()
print("Key: the step perceptron draws ONE straight line — cannot separate XOR.")
print("The MLP carves a non-linear boundary by composing multiple linear separations.")
print("This is WHY we need MLPs for digit classification in Exercises 2 and 3.")

---
## Summary

| Model | Problem | Result | Key takeaway |
|---|---|---|---|
| Step Perceptron | AND | Converges | Works for linearly separable |
| Linear Perceptron | y = x | Fits correctly | Delta rule verified |
| Non-linear (tanh) | y = tanh(x) | Fits the shape | Chain rule works |
| Non-linear (ReLU) | y = tanh(x) | Rough fit | Piecewise linear only |
| Step Perceptron | XOR | Fails | One hyperplane insufficient |
| MLP [2,2,1] | XOR | Solved | Backprop enables non-linear separation |
| MLP [2,3,2,1] | XOR | Solved | Deeper = richer boundary |

> **Bridge to Exercise 1:** We will use `NonLinearPerceptron` as TinyModel for fraud detection.  
> Key question: which activation is right when output must be a **probability in [0,1]**?
> 
> **Bridge to Exercises 2/3:** We will use `MLP` for digit classification —  
> the same class, scaled to larger architectures and real image data.